In [1]:
import numpy as np
import os

dir = "/scratch/gpfs/rk1593/clustering_output/"
X_list = []
searchlight_list = []
for index,file in enumerate(os.scandir(dir + "searchlights_collapseTR/")):
    file_name = file.name
    if ".csv" in file_name:
        if file_name == "64_30_33.csv":
            print("skipped 64_30_33.csv")
            continue
        if "NOT480" in file_name:
            print("Error: NOT480 ", file_name)
        if index % 10000 == 0:
            print(index)
        light_id = file_name.split(".")[0]
        features = np.genfromtxt(dir + "searchlights_collapseTR/" + file_name, delimiter = ",").tolist()
        searchlight_list += [light_id]
        X_list += [features]

print("extracted for /searchlights/ directory")
X = np.vstack(X_list)
print(X.shape)
np.save(file = dir + "kmeans_assignments_tval_collapseTR/master_X_list_collapseTR.npy", arr = X)
np.save(file = dir + "kmeans_assignments_tval_collapseTR/master_searchlight_collapseTR.npy", arr = np.array(searchlight_list))



0
10000
20000
30000
40000
50000
60000
70000
80000
90000
skipped 64_30_33.csv
100000
110000
120000
130000
140000
150000
160000
170000
180000
extracted for /searchlights/ directory
(185404, 54)


FileNotFoundError: [Errno 2] No such file or directory: '/scratch/gpfs/rk1593/clustering_output/kmeans_assignments_tval_collapseTR_confirm/master_X_list_collapseTR.npy'

In [2]:
np.save(file = dir + "kmeans_assignments_tval_collapseTR/master_X_list_collapseTR.npy", arr = X)
np.save(file = dir + "kmeans_assignments_tval_collapseTR/master_searchlight_collapseTR.npy", arr = np.array(searchlight_list))


In [3]:
dir

'/scratch/gpfs/rk1593/clustering_output/'

In [4]:
len(searchlight_list)

185404

In [5]:

import pickle
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans


def kmeans_clustering(dir, K):
    # read in the master_X_list and master_searchlight so that we have the cluster assignments
    # for the searchlights without having to output so much data each time
    # by outputting the master_X_list to the output csv multiple times
    # which would be a waste of della's memory
    X = np.load(file = dir + "master_X_list_collapseTR.npy")
    searchlight_list = np.load(file = dir + "master_searchlight_collapseTR.npy").tolist()
    print("K_clusters: ", K)
    kmeans = KMeans(n_clusters=K)
    cluster_labels = kmeans.fit_predict(X)
    # add 1 so that we have no label with zero
    cluster_labels += 1
    # create csv with the cluster assignment for this number of clusters
    # create the output df by giving cluster labels
    this_cost = kmeans.inertia_
    cost_list = [this_cost for i in range(len(searchlight_list))]
    output_dict = {"cluster_assignment": cluster_labels.tolist(),
                "searchlight": searchlight_list,
                "cost":cost_list
                }
    df = pd.DataFrame(output_dict)
    df.to_csv(dir + "kmeans_" + str(K) + "clusters_collapseTR.csv")
    # It is important to use binary access
    with open(dir + "kmeans_" + str(K) + "clusters_collapseTR.pickle", 'wb') as f:
        pickle.dump(kmeans, f)
  
for job_id_in in range(4,11):
    #job_id_in = 4#int(os.environ["SLURM_ARRAY_TASK_ID"])
    dir = "/scratch/gpfs/rk1593/clustering_output/kmeans_assignments_tval_collapseTR/"
    K = job_id_in
    kmeans_clustering(dir = dir, K = K)

K_clusters:  4
K_clusters:  5
K_clusters:  6
K_clusters:  7
K_clusters:  8
K_clusters:  9
K_clusters:  10


In [9]:
    dir = "/scratch/gpfs/rk1593/clustering_output/kmeans_assignments_tval/"
X = np.load(file = dir + "master_X_list_tval.npy")
X.shape

(185405, 774)

In [10]:
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_samples
import os
import pandas as pd

num_jobs = 4
dir = "/scratch/gpfs/rk1593/clustering_output/" 
#job_id_in = int(os.environ["SLURM_ARRAY_TASK_ID"])
for job_id_in in [2,3]:
    K = job_id_in
    labels_dir = "/scratch/gpfs/rk1593/clustering_output/kmeans_assignments_collapseTR/kmeans_" + str(K) + "clusters_collapseTR.csv"
    df = pd.read_csv(labels_dir)
    labels = df["cluster_assignment"].tolist()
    df = 0 # for memory
    output_dir = "/scratch/gpfs/rk1593/clustering_output/kmeans_silhouette_collapseTR/kmeans" + str(K) + "_silhouttes_collapseTR.csv"
    X = np.load("/scratch/gpfs/rk1593/clustering_output/kmeans_assignments_collapseTR/master_X_list_collapseTR.npy")
    X = X.astype("float32") # for memory limits
    print("got X")
    D_matrix = pairwise_distances(X, n_jobs= num_jobs)
    print("got D_matrix")
    X = 0
    sample_silhouette_values = silhouette_samples(X = D_matrix, labels = labels, metric = 'precomputed')
    print("got s_coeffs")
    np.savetxt(output_dir, sample_silhouette_values, delimiter=",")


got X


KeyboardInterrupt: 